# 5. Release: the model on Hugging Face and the demo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chingkheinganba231005/Meitei-Mayek-Diffusion-model/blob/main/notebooks/5_release.ipynb)

**Runtime: CPU.**

1. **Speed on a CPU** (E7): seconds a word on 2 threads, as on the free Space hardware.
2. **Export**: `generator.safetensors` (the chosen average, float16) and `config.json`, the
   adapted `owner_generator.safetensors` if it was better on the held-out words, the cards
   filled in from the results files, the presets (six test writers and the author's four
   references), and the Space with the code it runs (`mayek_diffusion`, and `mayek_words` with
   its font).
3. **A check of the Space**: the app writes two words in each kind of hand.
4. **Upload** (`UPLOAD = True`): the model repository and the Space.

Before uploading, check that the terms of TUMMHCD allow releasing a generator trained on its
images (the recogniser trained on them is already public), and choose the licence of the
weights (`LICENCE`; the recogniser's is MIT). The upload needs a Hugging Face token with write
access as a Colab secret called `HF_TOKEN` (key icon on the left; switch on notebook access),
or typed when asked. Never paste the token into a cell.

In [ ]:
import os

# Where things are. Change these to match your Drive.
WORK = "/content/drive/MyDrive/meitei-word-diffusion"         # this project's working folder
WORK_OLD = "/content/drive/MyDrive/meitei-word-recognition"   # the word-recognition project's folder
REPO = "chingkheinganba231005/Meitei-Mayek-Diffusion-model"
BRANCH = "main"
WORD_REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
WORD_COMMIT = "bdee9e7"                                       # the synthesiser this project builds on
CONTENT = "/content"                                          # the runtime's own disk

HF_ID = "Chingkheinganba/handwritten-meitei-mayek-word-generation"   # model repository and Space
LICENCE = "mit"            # of the weights
UPLOAD = False             # True to upload the model and the Space
PRIVATE = False

DISCONNECT_AT_END = True    # the last cell disconnects the runtime (Colab bills a connected runtime)

In [ ]:
import json, re, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
os.chdir(CONTENT)
CODE, WORD = f"{CONTENT}/code", f"{CONTENT}/word_repo"


def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    if (r.stdout + r.stderr).strip():
        print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")


def stream(*args):
    """Runs a long command, showing its output as it comes; stops the notebook if it fails.
    Returns the output."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    if p.wait():
        raise SystemExit(f"exit code {p.returncode}: {' '.join(str(a) for a in args)[:200]}")
    return "".join(lines)


def clone(url, path, ref):
    """A fresh copy of a repository at a branch or a commit."""
    shutil.rmtree(path, ignore_errors=True)
    run("git", "clone", "-q", url, path)
    run("git", "-C", path, "checkout", "-q", ref)


clone(f"https://github.com/{REPO}.git", CODE, BRANCH)
clone(f"https://github.com/{WORD_REPO}.git", WORD, WORD_COMMIT)
sys.path[:0] = [CODE, WORD]
os.chdir(CODE)

run(sys.executable, "-m", "pip", "install", "-q", "diffusers==0.41.0", "safetensors", "huggingface_hub", "timm")

for d in ("data", "runs", "results", "eval"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
run("git", "-C", CODE, "log", "-1", "--format=code: %h %s")

WEIGHTS, RELEASE = f"{WORK}/runs/main/final.pt", f"{WORK}/release"
if not os.path.exists(WEIGHTS):
    raise SystemExit(f"{WEIGHTS} does not exist yet: finish notebook 2 first")

## 1. Speed on a CPU

In [ ]:
import platform
import numpy as np
import torch
from mayek_diffusion.data import WordSet
from mayek_diffusion.protocol import chosen_setting
from mayek_diffusion.release import SAMPLING, load_weights
from mayek_diffusion.sample import generate
from mayek_diffusion.writers import Printer
from mayek_words.synth import load_priors

torch.set_num_threads(2)
grid = f"{WORK}/results/eval_val_grid.json"
setting = chosen_setting(grid if os.path.exists(grid) else None)
model, _ = load_weights(WEIGHTS, setting["sigma_rel"])
printer = Printer(load_priors())
test = WordSet(f"{WORK}/data/test")
refs = [test.target(i).astype(np.float32) / 255 for i in range(4)]
texts = [str(test.texts[i]) for i in range(4, 10)]
kw = dict(steps=SAMPLING["demo_steps"], sampler=SAMPLING["demo_sampler"], guidance=setting["guidance"],
          interval=tuple(setting["interval"]))
generate(model, texts[:1], refs, printer, **kw)          # warm-up
times = []
for t in texts:
    t0 = time.time()
    generate(model, [t], refs, printer, **kw)
    times.append(round(time.time() - t0, 2))
speed = {"threads": 2, "steps": kw["steps"], "sampler": kw["sampler"], "guidance": kw["guidance"],
         "interval": list(kw["interval"]), "seconds_per_word": round(float(np.mean(times)), 2), "seconds": times,
         "texts": texts, "cpu": platform.processor() or platform.machine(), "cores": os.cpu_count()}
Path(f"{WORK}/results/speed_cpu.json").write_text(json.dumps(speed, indent=1, ensure_ascii=False))
print(json.dumps(speed, indent=1, ensure_ascii=False))
del model

## 2. Export

In [ ]:
args = ["--word-repo", WORD, "--weights", WEIGHTS, "--results", f"{WORK}/results", "--data", f"{WORK}/data",
        "--out", RELEASE, "--hf-id", HF_ID, "--licence", LICENCE]
if os.path.exists(f"{WORK}/runs/owner/owner.pt"):
    args += ["--owner-weights", f"{WORK}/runs/owner/owner.pt"]
stream(sys.executable, "-u", "scripts/export.py", *args)
for d in ("model", "space"):
    print(f"\n{d}:")
    for p in sorted(Path(f"{RELEASE}/{d}").iterdir()):
        size = sum(f.stat().st_size for f in p.rglob("*")) if p.is_dir() else p.stat().st_size
        print(f"  {p.name}{'/' if p.is_dir() else ''}  {size / 1e6:.2f} MB")
print("\n" + Path(f"{RELEASE}/model/README.md").read_text(encoding="utf-8"))

## 3. A check of the Space

The app as the Space runs it (from the exported folder instead of Hugging Face): two words in
each kind of hand, 14 steps, on 2 threads.

In [ ]:
run(sys.executable, "-m", "pip", "install", "-q", "gradio")
check = """
import app
for source, preset in ((app.CHOICES[0], None), (app.PRESET, list(app.PRESETS)[0]), (app.RANDOM, None)):
    image, path, notes = app.write("ꯃꯤꯇꯩ ꯃꯌꯦꯛ", source, preset, None, 2.0, 14, 1.0, 0, 1)
    print(source, image.shape, notes)
app.build()
print("the Space works")
"""
r = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True, cwd=f"{RELEASE}/space",
                   env=dict(os.environ, MODEL_DIR=f"{RELEASE}/model"))
print(r.stdout[-3000:], r.stderr[-3000:])
if r.returncode:
    raise SystemExit("the Space failed: look at the error above")

## 4. Upload

In [ ]:
if not UPLOAD:
    print("UPLOAD is False: nothing is uploaded.")
else:
    from huggingface_hub import HfApi
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        import getpass
        hf_token = getpass.getpass("Hugging Face token with write access: ")
    api = HfApi(token=hf_token)
    api.create_repo(HF_ID, repo_type="model", exist_ok=True, private=PRIVATE)
    api.create_repo(HF_ID, repo_type="space", space_sdk="gradio", exist_ok=True, private=PRIVATE)
    api.upload_folder(folder_path=f"{RELEASE}/model", repo_id=HF_ID, repo_type="model",
                      commit_message="The generator, its configuration, results and card")
    api.upload_folder(folder_path=f"{RELEASE}/space", repo_id=HF_ID, repo_type="space",
                      commit_message="Demo")
    print(f"model: https://huggingface.co/{HF_ID}")
    print(f"demo:  https://huggingface.co/spaces/{HF_ID}")

## Disconnect

Colab bills a connected runtime whether or not it is working. The cell below disconnects it
(`DISCONNECT_AT_END = True`); otherwise use *Runtime > Disconnect and delete runtime*. Every
result is already on Drive.

In [ ]:
# Colab bills a connected runtime whether or not it is working: disconnect it now.
if DISCONNECT_AT_END:
    from google.colab import runtime
    runtime.unassign()
else:
    print("Finished. Disconnect the runtime: Runtime > Disconnect and delete runtime.")